In [3]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [4]:
def detect_mouth(image):
    """
    Applies face and mouth detection on an image.
    Returns the image with drawn rectangles and the coordinates of the detected mouth.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)

    # Load cascades (ensure the paths are correct)
    face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
    
    if len(faces) == 0:
        return image, None
    
    (x, y, w, h) = faces[0]
    cv2.rectangle(image, (x, y), (x + w, y + h), (255, 0, 0), 2)
    
    mouth_roi_y_start = int(y + 0.6 * h)
    mouth_roi = gray[mouth_roi_y_start: y + h, x: x + w]
  
    mouth_cascade = cv2.CascadeClassifier('haarcascade_mcs_mouth.xml')
    mouths = mouth_cascade.detectMultiScale(mouth_roi, scaleFactor=1.1, minNeighbors=5)
    
    if len(mouths) == 0:
        return image, None

    best_candidate = None
    best_y = -1
    for (mx, my, mw, mh) in mouths:
        absolute_y = mouth_roi_y_start + my 
        if absolute_y > best_y:
            best_y = absolute_y
            best_candidate = (mx, my, mw, mh)

    if best_candidate is None:
        return image, None

    mx, my, mw, mh = best_candidate
    mouth_x = x + mx
    mouth_y = mouth_roi_y_start + my
    cv2.rectangle(image, (mouth_x, mouth_y), (mouth_x + mw, mouth_y + mh), (0, 255, 0), 2)
    detected_mouth = (mouth_x, mouth_y, mw, mh)

    return image, detected_mouth

In [ ]:
def extract_mouth_roi(video_path, target_size=(112,112)):
    """
    For a given video, extracts the sequence of frames.
    For each frame, uses detect_mouth to get the mouth ROI.
    If a mouth is detected, crops the frame to that area,
    converts the cropped image to grayscale, and resizes it to target_size.
    Returns a numpy array of shape (T, H, W) where T is the number of frames extracted.
    """
    cap = cv2.VideoCapture(video_path)
    roi_frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # Apply detection on the frame
        _, mouth_coords = detect_mouth(frame)
        if mouth_coords is not None:
            (mx, my, mw, mh) = mouth_coords
            # Crop the ROI from the original color image and convert to grayscale
            roi = frame[my:my+mh, mx:mx+mw]
            roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
            # Resize to target_size
            roi_resized = cv2.resize(roi_gray, target_size)
            roi_frames.append(roi_resized)
        else:
            # Skip frame if no mouth is detected
            continue
    cap.release()
    return np.array(roi_frames)

def frames_to_tensor_tf(frames_roi):
    """
    Converts a sequence of frames (numpy array of shape (T, H, W)) into a TF tensor
    of shape (T, H, W, 1) and normalizes pixel values between 0 and 1.
    """
    # Add a channel dimension
    frames_roi = frames_roi[..., np.newaxis]  # (T, H, W, 1)
    frames_tensor = frames_roi.astype(np.float32) / 255.0
    return frames_tensor

In [4]:
def dataset_generator(split="train"):

    dataset_root = "lipread_mp4"  # Adapt this path to your data organization
    # List of words (classes)
    words = sorted([d for d in os.listdir(dataset_root) if os.path.isdir(os.path.join(dataset_root, d))])
    word_to_idx = {word: idx for idx, word in enumerate(words)}
    
    for word in words:
        folder = os.path.join(dataset_root, word, split)
        if not os.path.exists(folder):
            continue
        for file in os.listdir(folder):
            if file.endswith(".mp4"):
                video_path = os.path.join(folder, file)
                frames_roi = extract_mouth_roi(video_path, target_size=(112,112))
                if frames_roi.size == 0:
                    continue
                video_tensor = frames_to_tensor_tf(frames_roi)  # shape (T, 112, 112, 1)
                label = word_to_idx[word]
                yield video_tensor, label

def create_dataset(split="train", batch_size=8, target_frames=29):
    """
    Creates a tf.data.Dataset from the generator.
    Uses padded_batch to obtain a fixed number of frames (target_frames).
    """
    output_types = (tf.float32, tf.int32)
    output_shapes = (tf.TensorShape([None, 112, 112, 1]), tf.TensorShape([]))
    
    ds = tf.data.Dataset.from_generator(
        lambda: dataset_generator(split),
        output_types=output_types,
        output_shapes=output_shapes
    )
    ds = ds.padded_batch(batch_size, padded_shapes=([target_frames, 112, 112, 1], []))
    ds = ds.prefetch(tf.data.experimental.AUTOTUNE)
    return ds

# Create datasets for train, validation, and test splits
train_ds = create_dataset(split="train", batch_size=8, target_frames=29)
val_ds   = create_dataset(split="val", batch_size=8, target_frames=29)
test_ds  = create_dataset(split="test", batch_size=8, target_frames=29)

NameError: name 'tf' is not defined

In [3]:
train_ds

NameError: name 'train_ds' is not defined